# Stage 03 · Step 0 — Env wiring + reward sanity

Build the `MaintenanceBanditEnv` on top of the frozen Stage 02 PatchTST encoder, then run 100 random τ vectors on a handful of train printers and verify the reward distribution looks like Stage 01's trial-cost distribution. If the histograms are wildly different, the env is wired wrong — fix it before training PPO.

If `02_ssl/models/ssl_encoder.pt` is missing this notebook falls back to a randomly-initialised encoder so it still runs end-to-end (with uninformative embeddings).

In [ ]:
from __future__ import annotations
import warnings

import matplotlib.pyplot as plt
import numpy as np

from ml_models import PROJECT_ROOT
from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TRAIN_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.env_runner import default_dates
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.rl import (
    MaintenanceBanditEnv,
    TAU_RANGES,
    action_to_tau,
    load_ssl_encoder,
    random_encoder_bundle,
)
from sdg.generate import load_configs

RESULTS_DIR = PROJECT_ROOT / 'ml_models/03_rl+ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Try to load the Stage 02 encoder; fall back to random init if it doesn't
# exist yet. The fallback is for sanity testing only — train/eval should run
# the real encoder.
fleet = load_fleet(DEFAULT_FLEET_PATH)
subset = filter_printers(fleet, list(TRAIN_PRINTERS[:8]) + list(VAL_PRINTERS[:2]))
_, feature_cols = build_feature_matrix(subset)

try:
    bundle = load_ssl_encoder()
    print(f'Loaded Stage 02 encoder — d_model={bundle.d_model}, channels={len(bundle.feature_columns)}')
except FileNotFoundError as exc:
    warnings.warn(f'{exc}\nFalling back to random-init encoder — results will not show the SSL benefit.')
    bundle = random_encoder_bundle(feature_columns=feature_cols, d_model=64, device='cpu')
    print(f'Using random encoder — d_model={bundle.d_model}')

In [ ]:
components_cfg, couplings_cfg, cities_cfg = load_configs()
DATES = default_dates()

# 5-printer mini env covering several cities. Each step runs the full
# 10-year simulation on one printer, so 100 random trials ≈ 100 simulator
# calls — a couple of minutes on CPU.
SANITY_PRINTERS = list(TRAIN_PRINTERS[:5])
env = MaintenanceBanditEnv(
    printer_ids=SANITY_PRINTERS,
    encoder_bundle=bundle,
    components_cfg=components_cfg,
    couplings_cfg=couplings_cfg,
    cities_cfg=cities_cfg,
    dates=DATES,
)
print('obs_space:', env.observation_space)
print('action_space:', env.action_space)
print('observation dim:', env.obs_dim)

## Random-action reward distribution

Sample 100 actions uniformly in $[-1, 1]^6$, run them through `step`, and collect (`annual_cost`, `availability`, `reward`). The reward histogram should be unimodal and dispersed across orders of magnitude — if everything clusters at a single value the env is broken.

In [ ]:
rng = np.random.default_rng(0)
N_TRIALS = 100

rewards = []
annual_costs = []
availabilities = []
deficits = []
for trial in range(N_TRIALS):
    pid = SANITY_PRINTERS[trial % len(SANITY_PRINTERS)]
    env.reset(seed=int(trial), options={'printer_id': pid})
    action = rng.uniform(-1.0, 1.0, size=6).astype(np.float32)
    _, reward, _, _, info = env.step(action)
    rewards.append(reward)
    annual_costs.append(info['annual_cost'])
    availabilities.append(info['availability'])
    deficits.append(info['deficit'])
    if trial % 10 == 0:
        print(f'trial {trial:3d}  pid={pid:3d}  cost={info["annual_cost"]:.2e}  avail={info["availability"]:.3f}  reward={reward:.3f}')

print()
print(f'reward      mean={np.mean(rewards):.3f}  min={np.min(rewards):.3f}  max={np.max(rewards):.3f}')
print(f'annual_cost mean={np.mean(annual_costs):.3e}  median={np.median(annual_costs):.3e}')
print(f'availability mean={np.mean(availabilities):.3f}  feasible_frac={np.mean(np.array(availabilities) >= 0.95):.2%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(rewards, bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Reward distribution under random τ')
axes[0].set_xlabel('reward (=-(cost/1e6 + 100·deficit))')
axes[0].set_ylabel('count')

axes[1].hist(annual_costs, bins=20, color='salmon', edgecolor='black')
axes[1].set_title('Per-printer annual cost (€/yr)')
axes[1].set_xlabel('annual_cost')
axes[1].set_ylabel('count')
axes[1].set_xscale('log')

axes[2].scatter(annual_costs, availabilities, alpha=0.6, color='seagreen')
axes[2].axhline(0.95, color='red', linestyle='--', label='95% threshold')
axes[2].set_xscale('log')
axes[2].set_xlabel('annual_cost')
axes[2].set_ylabel('availability')
axes[2].set_title('Trial Pareto')
axes[2].legend()

plt.tight_layout()
plt.show()

## Sanity check: action bounds map to expected τ values

All-`-1` action should give the lower per-component bound; all-`+1` the upper bound.

In [ ]:
low_tau = action_to_tau(-np.ones(6, dtype=np.float32))
high_tau = action_to_tau(np.ones(6, dtype=np.float32))
centre_tau = action_to_tau(np.zeros(6, dtype=np.float32))
import pandas as pd
table = pd.DataFrame({
    'low (a=-1)': low_tau,
    'centre (a=0)': centre_tau,
    'high (a=+1)': high_tau,
    'TAU_RANGES.low': {c: lo for c, (lo, _) in TAU_RANGES.items()},
    'TAU_RANGES.high': {c: hi for c, (_, hi) in TAU_RANGES.items()},
})
table.round(2)

## Acceptance

- [x] Reward histogram is dispersed (multi-modal across cost magnitudes).
- [x] At least one random trial achieves availability ≥ 0.95 (so the policy class is feasible at all).
- [x] Action bounds map to the expected τ ranges per component.

If the histogram is a single spike or all trials are infeasible, do not move on — fix the env wiring first.